# ⭐ Day 82: Model Interpretability & Production Pipeline for Customer Churn Prediction
## Day 82 of 369-day Python & AI Learning Path

🏆 **Welcome to Day 82!** Today we transform our churn prediction model from a "black box" into a **transparent, interpretable, and production-ready** asset that business stakeholders can trust and act upon.

## 🚀 Introduction

In the previous days, we built powerful machine learning models to predict customer churn. But here's the critical question: **"Why did the model predict this customer will churn?"**

Today, we master:
- 💡 **SHAP (SHapley Additive exPlanations)** — the gold standard for model interpretability
- 📊 **Global & Local Interpretability** — understanding the big picture and individual predictions
- 🏗️ **Production Pipelines** — building robust, reproducible ML systems
- 🚀 **Model Serialization & Inference** — deploying models to create real business value
- 📈 **Monitoring & Logging** — keeping our models healthy in production

Let's make our models **explainable, actionable, and production-ready!**

In [ ]:
# 📦 Standard Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 🎨 Visualization Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

# 🤖 ML & Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, precision_recall_curve)

# 🔍 SHAP for Interpretability
import shap
shap.initjs()

# 💾 Serialization
import joblib
import json
import logging
from datetime import datetime
import os

print("✅ All libraries imported successfully!")
print(f"📅 Session Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📂 Loading the Best Model and Engineered Dataset

In [ ]:
# 📂 Load the feature-engineered dataset
data_path = '/kaggle/input/datasets/abbas829/telecom-customer-churn-feature-engineering-dataset/telecom_customer_churn_feature_engineering.csv'

try:
    df = pd.read_csv(data_path)
    print(f"✅ Dataset loaded successfully!")
    print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\n🔍 First 5 rows:")
    display(df.head())
except FileNotFoundError:
    print(f"❌ File not found at {data_path}")
    print("📝 Creating a synthetic telecom churn dataset for demonstration...")
    
    # Create synthetic dataset matching typical telecom churn features
    np.random.seed(42)
    n_samples = 7043  # Typical Telco dataset size
    
    df = pd.DataFrame({
        'customerID': [f'CUST_{i:05d}' for i in range(n_samples)],
        'gender': np.random.choice(['Male', 'Female'], n_samples),
        'SeniorCitizen': np.random.choice([0, 1], n_samples, p=[0.85, 0.15]),
        'Partner': np.random.choice(['Yes', 'No'], n_samples, p=[0.5, 0.5]),
        'Dependents': np.random.choice(['Yes', 'No'], n_samples, p=[0.3, 0.7]),
        'tenure': np.random.randint(0, 72, n_samples),
        'PhoneService': np.random.choice(['Yes', 'No'], n_samples, p=[0.9, 0.1]),
        'MultipleLines': np.random.choice(['Yes', 'No', 'No phone service'], n_samples),
        'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.35, 0.45, 0.2]),
        'OnlineSecurity': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'OnlineBackup': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'DeviceProtection': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'TechSupport': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'StreamingTV': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'StreamingMovies': np.random.choice(['Yes', 'No', 'No internet service'], n_samples),
        'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.55, 0.25, 0.2]),
        'PaperlessBilling': np.random.choice(['Yes', 'No'], n_samples, p=[0.6, 0.4]),
        'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'], n_samples),
        'MonthlyCharges': np.random.uniform(18, 118, n_samples).round(2),
        'TotalCharges': np.random.uniform(18, 8684, n_samples).round(2),
        'Churn': np.random.choice(['Yes', 'No'], n_samples, p=[0.27, 0.73])
    })
    
    # Add engineered features
    df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-12', '13-24', '25-48', '49-72'])
    df['avg_monthly_charges'] = (df['TotalCharges'] / (df['tenure'] + 1)).round(2)
    df['services_count'] = (df[['PhoneService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']] == 'Yes').sum(axis=1)
    df['is_new_customer'] = (df['tenure'] <= 12).astype(int)
    df['high_value_customer'] = ((df['MonthlyCharges'] > 80) & (df['tenure'] > 24)).astype(int)
    
    print(f"✅ Synthetic dataset created!")
    print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    display(df.head())

In [ ]:
# 🔍 Quick data inspection
print("📋 Column Names and Data Types:")
print(df.dtypes)
print(f"\n🔢 Missing Values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\n🎯 Target Distribution:")
print(df['Churn'].value_counts(normalize=True).round(3))

In [ ]:
# 🏗️ Prepare features and target
target_col = 'Churn'
id_col = 'customerID'

# Separate features and target
X = df.drop(columns=[target_col, id_col], errors='ignore')
y = df[target_col].map({'Yes': 1, 'No': 0})

# Identify column types
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"🔢 Numeric Features ({len(numeric_features)}): {numeric_features}")
print(f"📊 Categorical Features ({len(categorical_features)}): {categorical_features}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✅ Data split complete:")
print(f"📈 Training set: {X_train.shape[0]:,} samples")
print(f"📉 Test set: {X_test.shape[0]:,} samples")
print(f"🎯 Churn rate in train: {y_train.mean():.2%}")
print(f"🎯 Churn rate in test: {y_test.mean():.2%}")

In [ ]:
# 🏆 Build and train the best model (Gradient Boosting for interpretability + performance)
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

# Build the full pipeline with Gradient Boosting
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=4,
        min_samples_split=20,
        min_samples_leaf=10,
        subsample=0.8,
        random_state=42
    ))
])

print("🚀 Training the production-ready model...")
best_pipeline.fit(X_train, y_train)

# Evaluate
train_auc = roc_auc_score(y_train, best_pipeline.predict_proba(X_train)[:, 1])
test_auc = roc_auc_score(y_test, best_pipeline.predict_proba(X_test)[:, 1])

print(f"\n🏆 Model Performance:")
print(f"📈 Train AUC: {train_auc:.4f}")
print(f"📉 Test AUC:  {test_auc:.4f}")
print(f"✅ Model trained and ready for interpretability analysis!")

## 🔍 Global Interpretability with SHAP

**Global interpretability** helps us understand which features drive churn predictions across our entire customer base. This is crucial for strategic decision-making!

In [ ]:
# 🎯 Prepare data for SHAP analysis
# Get preprocessed feature names
preprocessor_fitted = best_pipeline.named_steps['preprocessor']
feature_names = (numeric_features + 
                 list(preprocessor_fitted.named_transformers_['cat'].get_feature_names_out(categorical_features)))

# Transform test data
X_test_processed = preprocessor_fitted.transform(X_test)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print(f"✅ Preprocessed test data shape: {X_test_processed_df.shape}")
print(f"📊 Sample features: {feature_names[:10]}...")

In [ ]:
# 💡 Calculate SHAP values using TreeExplainer (optimized for tree-based models)
print("🧮 Computing SHAP values... (this may take a moment)")
explainer = shap.TreeExplainer(best_pipeline.named_steps['classifier'])
shap_values = explainer.shap_values(X_test_processed)

print(f"✅ SHAP values computed!")
print(f"📊 SHAP values shape: {np.array(shap_values).shape}")
print(f"💡 For binary classification, we analyze SHAP values for class 1 (Churn)")

In [ ]:
# 📊 SHAP Summary Plot — Global Feature Importance
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values[1] if isinstance(shap_values, list) else shap_values, 
    X_test_processed_df,
    feature_names=feature_names,
    show=False,
    max_display=20
)
plt.title('🔍 SHAP Summary Plot: Global Feature Importance for Churn Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Interpretation Guide:")
print("   • Each dot = one customer")
print("   • Color (red=high, blue=low) = feature value")
print("   • Horizontal position = impact on churn probability")
print("   • Features sorted by total impact (importance)")

In [ ]:
# 📊 SHAP Bar Plot — Mean Absolute Impact (Feature Importance Ranking)
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values[1] if isinstance(shap_values, list) else shap_values,
    X_test_processed_df,
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    max_display=15
)
plt.title('🏆 Top 15 Features by Mean |SHAP Value| (Global Importance)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate and display exact importance values
shap_importance = pd.DataFrame({
    'feature': feature_names,
    'mean_shap_value': np.abs(shap_values[1] if isinstance(shap_values, list) else shap_values).mean(axis=0)
}).sort_values('mean_shap_value', ascending=False)

print("📊 Top 10 Most Important Features for Churn Prediction:")
print(shap_importance.head(10).to_string(index=False))

In [ ]:
# 📈 Feature Importance Comparison: SHAP vs Built-in
# Get built-in feature importance from Gradient Boosting
gb_model = best_pipeline.named_steps['classifier']
builtin_importance = pd.DataFrame({
    'feature': feature_names,
    'builtin_importance': gb_model.feature_importances_
}).sort_values('builtin_importance', ascending=False)

# Merge for comparison
comparison = shap_importance.merge(builtin_importance, on='feature').head(15)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# SHAP Importance
axes[0].barh(comparison['feature'][::-1], comparison['mean_shap_value'][::-1], color='steelblue')
axes[0].set_title('💡 SHAP Mean |Impact|', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Mean |SHAP Value|')

# Built-in Importance
axes[1].barh(comparison['feature'][::-1], comparison['builtin_importance'][::-1], color='coral')
axes[1].set_title('🔧 Built-in Feature Importance', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.suptitle('📊 Feature Importance Comparison: SHAP vs Built-in', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 Key Insight: SHAP provides more reliable importance by considering feature interactions and actual contribution to predictions!")

## 🎯 Local Interpretability with SHAP Force Plots

**Local interpretability** explains individual predictions — essential for customer service teams and personalized retention campaigns!

In [ ]:
# 🎯 Select interesting customers for local explanation
# Find a high-risk customer and a low-risk customer
proba = best_pipeline.predict_proba(X_test)[:, 1]
risk_df = X_test.copy()
risk_df['churn_probability'] = proba
risk_df['actual_churn'] = y_test.values

# High-risk customer (predicted to churn with high confidence)
high_risk_idx = risk_df[risk_df['churn_probability'] > 0.8].index[0]
# Low-risk customer (predicted to stay with high confidence)  
low_risk_idx = risk_df[risk_df['churn_probability'] < 0.1].index[0]
# Medium-risk customer
medium_risk_idx = risk_df[(risk_df['churn_probability'] > 0.45) & (risk_df['churn_probability'] < 0.55)].index[0]

print("🎯 Selected Customers for Local Interpretability:")
print(f"🔴 High-Risk Customer (ID: {high_risk_idx}): {risk_df.loc[high_risk_idx, 'churn_probability']:.2%} churn probability")
print(f"🟢 Low-Risk Customer (ID: {low_risk_idx}): {risk_df.loc[low_risk_idx, 'churn_probability']:.2%} churn probability")
print(f"🟡 Medium-Risk Customer (ID: {medium_risk_idx}): {risk_df.loc[medium_risk_idx, 'churn_probability']:.2%} churn probability")

In [ ]:
# 🔴 Force Plot for High-Risk Customer
print("🔴 HIGH-RISK CUSTOMER — Why will they likely churn?")
print("="*60)

# Get SHAP values for this customer
idx_position = list(X_test.index).index(high_risk_idx)
shap_values_class1 = shap_values[1] if isinstance(shap_values, list) else shap_values

# Create force plot
shap.force_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    shap_values_class1[idx_position],
    X_test_processed_df.iloc[idx_position],
    feature_names=feature_names,
    matplotlib=True,
    show=False
)
plt.title(f'🔴 High-Risk Customer (Churn Prob: {risk_df.loc[high_risk_idx, "churn_probability"]:.1%})', 
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Print key drivers
customer_shap = pd.DataFrame({
    'feature': feature_names,
    'value': X_test_processed_df.iloc[idx_position].values,
    'shap_value': shap_values_class1[idx_position]
}).sort_values('shap_value', key=abs, ascending=False)

print("\n📊 Top Drivers for this customer:")
print(customer_shap.head(8).to_string(index=False))

In [ ]:
# 🟢 Force Plot for Low-Risk Customer
print("🟢 LOW-RISK CUSTOMER — Why will they likely stay?")
print("="*60)

idx_position_low = list(X_test.index).index(low_risk_idx)

shap.force_plot(
    explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    shap_values_class1[idx_position_low],
    X_test_processed_df.iloc[idx_position_low],
    feature_names=feature_names,
    matplotlib=True,
    show=False
)
plt.title(f'🟢 Low-Risk Customer (Churn Prob: {risk_df.loc[low_risk_idx, "churn_probability"]:.1%})', 
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Print key retention factors
customer_shap_low = pd.DataFrame({
    'feature': feature_names,
    'value': X_test_processed_df.iloc[idx_position_low].values,
    'shap_value': shap_values_class1[idx_position_low]
}).sort_values('shap_value', key=abs, ascending=False)

print("\n📊 Top Retention Factors for this customer:")
print(customer_shap_low.head(8).to_string(index=False))

In [ ]:
# 🌊 Waterfall Plot for detailed breakdown (SHAP 0.40+ style)
print("🌊 Detailed SHAP Waterfall Plot for High-Risk Customer")
print("="*60)

try:
    # For newer SHAP versions
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_values_class1[idx_position],
            base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
            data=X_test_processed_df.iloc[idx_position].values,
            feature_names=feature_names
        ),
        max_display=15,
        show=False
    )
    plt.title('🌊 SHAP Waterfall: Feature Contribution Breakdown', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
except:
    print("💡 Waterfall plot requires SHAP 0.40+. Using bar plot instead...")
    plt.figure(figsize=(10, 8))
    top_features = customer_shap.head(15)
    colors = ['red' if v > 0 else 'green' for v in top_features['shap_value']]
    plt.barh(top_features['feature'][::-1], top_features['shap_value'][::-1], color=colors[::-1])
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.title('🌊 Feature Contribution to Churn Prediction', fontsize=12, fontweight='bold')
    plt.xlabel('SHAP Value (Red = Churn Risk, Green = Retention Factor)')
    plt.tight_layout()
    plt.show()

## 📋 Business-Friendly Interpretability Reports

Let's create reports that non-technical stakeholders can understand and act upon!

In [ ]:
# 🏢 Create Business-Friendly Interpretability Report
def generate_customer_report(customer_id, X_data, y_true, model, explainer, shap_vals, feature_names, raw_data):
    """
    Generate a business-friendly interpretability report for a single customer.
    """
    idx = list(X_data.index).index(customer_id)
    proba = model.predict_proba(X_data.loc[[customer_id]])[0, 1]
    prediction = "🔴 LIKELY TO CHURN" if proba > 0.5 else "🟢 LIKELY TO STAY"
    
    # Get SHAP values for this customer
    shap_vals_customer = shap_vals[1][idx] if isinstance(shap_vals, list) else shap_vals[idx]
    
    # Create explanation dataframe
    explanation = pd.DataFrame({
        'feature': feature_names,
        'shap_value': shap_vals_customer,
        'direction': ['⬆️ Increases Risk' if v > 0 else '⬇️ Decreases Risk' for v in shap_vals_customer]
    }).sort_values('shap_value', key=abs, ascending=False)
    
    # Top risk factors
    risk_factors = explanation[explanation['shap_value'] > 0].head(5)
    retention_factors = explanation[explanation['shap_value'] < 0].head(5)
    
    report = f"""
{'='*70}
📋 CUSTOMER CHURN RISK REPORT
{'='*70}
Customer ID: {customer_id}
Assessment Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}
{'='*70}

🎯 CHURN PREDICTION: {prediction}
📊 Churn Probability: {proba:.1%}
✅ Actual Outcome: {'Churned' if y_true.loc[customer_id] == 1 else 'Retained'}

{'='*70}
🔴 TOP 5 RISK FACTORS (Why this customer might churn):
{'='*70}
"""
    for _, row in risk_factors.iterrows():
        report += f"   {row['direction']:20s} | {row['feature']:30s} | Impact: +{row['shap_value']:.4f}\n"
    
    report += f"""
{'='*70}
🟢 TOP 5 RETENTION FACTORS (Why this customer might stay):
{'='*70}
"""
    for _, row in retention_factors.iterrows():
        report += f"   {row['direction']:20s} | {row['feature']:30s} | Impact: {row['shap_value']:.4f}\n"
    
    report += f"""
{'='*70}
💡 RECOMMENDED ACTIONS:
{'='*70}
"""
    # Generate actionable recommendations based on top risk factors
    recommendations = []
    for _, row in risk_factors.head(3).iterrows():
        feat = row['feature']
        if 'tenure' in feat.lower() and 'low' in feat.lower():
            recommendations.append("🎁 Offer welcome bonus or onboarding support to increase engagement")
        elif 'contract' in feat.lower() and 'month' in feat.lower():
            recommendations.append("📄 Propose contract upgrade with discount for longer commitment")
        elif 'monthlycharge' in feat.lower() or 'charge' in feat.lower():
            recommendations.append("💰 Review pricing plan — consider personalized discount or plan downgrade")
        elif 'service' in feat.lower() or 'internet' in feat.lower():
            recommendations.append("🔧 Proactive technical support and service quality check")
        elif 'payment' in feat.lower() and 'electronic' in feat.lower():
            recommendations.append("💳 Suggest automatic payment setup to reduce friction")
        elif 'support' in feat.lower() or 'security' in feat.lower():
            recommendations.append("🛡️ Offer premium support package or security features trial")
    
    if not recommendations:
        recommendations = [
            "📞 Schedule personalized retention call",
            "🎁 Offer loyalty reward or exclusive perk", 
            "📊 Conduct detailed satisfaction survey"
        ]
    
    for rec in set(recommendations):
        report += f"   • {rec}\n"
    
    report += f"{'='*70}\n"
    return report

# Generate report for high-risk customer
print(generate_customer_report(
    high_risk_idx, X_test, y_test, best_pipeline, 
    explainer, shap_values, feature_names, df
))

In [ ]:
# 📊 Batch Report Generation for Customer Service Team
def generate_batch_risk_report(X_data, model, top_n=20):
    """Generate a summary report for the top N highest-risk customers."""
    proba = model.predict_proba(X_data)[:, 1]
    risk_summary = X_data.copy()
    risk_summary['churn_probability'] = proba
    risk_summary = risk_summary.sort_values('churn_probability', ascending=False).head(top_n)
    
    print(f"{'='*80}")
    print(f"📊 TOP {top_n} CHURN RISK CUSTOMERS — ACTION REQUIRED")
    print(f"{'='*80}")
    print(f"{'Customer Segment':<20} {'Churn Prob':<12} {'Priority':<12} {'Suggested Action'}")
    print(f"{'-'*80}")
    
    for idx, (_, row) in enumerate(risk_summary.iterrows(), 1):
        prob = row['churn_probability']
        if prob > 0.8:
            priority = "🔴 CRITICAL"
            action = "Immediate retention call + executive outreach"
        elif prob > 0.6:
            priority = "🟠 HIGH"
            action = "Personalized offer within 48 hours"
        else:
            priority = "🟡 MEDIUM"
            action = "Targeted email campaign + survey"
        
        # Determine segment based on available features
        segment = "General"
        if 'tenure' in row:
            segment = "New" if row['tenure'] <= 12 else "Established" if row['tenure'] <= 36 else "Loyal"
        
        print(f"{segment:<20} {prob:<12.1%} {priority:<12} {action}")
    
    print(f"{'='*80}")
    print(f"💡 Total customers at risk (>50% probability): {(proba > 0.5).sum():,}")
    print(f"💰 Estimated revenue at risk: ${(proba * 100).sum():,.0f} (assuming $100 ARPU)")
    
    return risk_summary

# Generate batch report
top_risk_customers = generate_batch_risk_report(X_test, best_pipeline, top_n=15)

## 🏗️ Building a Production Pipeline

A **production pipeline** ensures our model can be reliably deployed, maintained, and scaled. Let's build a robust sklearn Pipeline!

In [ ]:
# 🏗️ Production-Grade Pipeline with Full Preprocessing
from sklearn.base import BaseEstimator, TransformerMixin

# Custom transformer for feature engineering
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Custom transformer for domain-specific feature engineering."""
    
    def __init__(self, create_interactions=True):
        self.create_interactions = create_interactions
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # Ensure numeric columns exist
        if 'tenure' in X.columns and 'MonthlyCharges' in X.columns:
            # Tenure groups
            X['tenure_group'] = pd.cut(
                X['tenure'], 
                bins=[0, 12, 24, 48, 72], 
                labels=['0-12', '13-24', '25-48', '49-72']
            ).astype(str)
            
            # Average monthly charges
            X['avg_monthly_charges'] = (X['TotalCharges'] / (X['tenure'] + 1)).round(2)
            
            # Customer lifecycle features
            X['is_new_customer'] = (X['tenure'] <= 12).astype(int)
            X['is_long_term'] = (X['tenure'] > 48).astype(int)
            
            # Value features
            X['high_value_customer'] = ((X['MonthlyCharges'] > 80) & (X['tenure'] > 24)).astype(int)
            X['low_value_at_risk'] = ((X['MonthlyCharges'] < 30) & (X['tenure'] < 12)).astype(int)
        
        # Service count if service columns exist
        service_cols = ['PhoneService', 'OnlineSecurity', 'OnlineBackup', 
                       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
        available_services = [col for col in service_cols if col in X.columns]
        if available_services:
            X['services_count'] = (X[available_services] == 'Yes').sum(axis=1)
        
        return X

# Build the complete production pipeline
production_pipeline = Pipeline([
    ('feature_engineer', FeatureEngineer()),
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features + ['avg_monthly_charges', 'services_count', 'is_new_customer', 'is_long_term', 'high_value_customer', 'low_value_at_risk']),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
        ]), categorical_features + ['tenure_group'])
    ], remainder='drop')),
    ('classifier', GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=4,
        min_samples_split=20,
        min_samples_leaf=10,
        subsample=0.8,
        random_state=42
    ))
])

print("🏗️ Production Pipeline Architecture:")
print("="*50)
for step_name, step in production_pipeline.steps:
    print(f"📌 {step_name}: {type(step).__name__}")
    if hasattr(step, 'transformers'):
        for name, transformer, cols in step.transformers:
            print(f"   └─ {name}: {type(transformer).__name__} on {len(cols)} features")
print("\n✅ Production pipeline defined successfully!")

In [ ]:
from sklearn.impute import SimpleImputer

# 🚀 Train the production pipeline on full dataset
print("🚀 Training production pipeline on full dataset...")

# Update feature lists for the engineered pipeline
X_full = df.drop(columns=[target_col, id_col], errors='ignore')
y_full = df[target_col].map({'Yes': 1, 'No': 0})

# Identify all features after engineering
numeric_features_full = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
                           'avg_monthly_charges', 'services_count', 'is_new_customer', 
                           'is_long_term', 'high_value_customer', 'low_value_at_risk']
categorical_features_full = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                             'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                             'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                             'PaperlessBilling', 'PaymentMethod', 'tenure_group']

# Filter to only available columns
numeric_features_full = [f for f in numeric_features_full if f in X_full.columns]
categorical_features_full = [f for f in categorical_features_full if f in X_full.columns]

# Rebuild pipeline with correct feature lists
production_pipeline = Pipeline([
    ('feature_engineer', FeatureEngineer()),
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features_full),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
        ]), categorical_features_full)
    ], remainder='drop')),
    ('classifier', GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=4,
        min_samples_split=20,
        min_samples_leaf=10,
        subsample=0.8,
        random_state=42
    ))
])

# Train on full data
production_pipeline.fit(X_full, y_full)

# Cross-validation score
cv_scores = cross_val_score(production_pipeline, X_full, y_full, cv=5, scoring='roc_auc')
print(f"\n🏆 Production Pipeline Performance:")
print(f"📊 5-Fold CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
print(f"✅ Pipeline is production-ready!")

## 💾 Model Serialization with Joblib

Serialization allows us to save and load models for deployment. **Joblib** is optimized for large numpy arrays!

In [ ]:
# 💾 Serialize the production pipeline
import os

# Create models directory
model_dir = './models'
os.makedirs(model_dir, exist_ok=True)

# Define model paths
model_path = os.path.join(model_dir, 'churn_prediction_pipeline.joblib')
preprocessor_path = os.path.join(model_dir, 'preprocessor.joblib')
metadata_path = os.path.join(model_dir, 'model_metadata.json')

# Save the complete pipeline
joblib.dump(production_pipeline, model_path)
print(f"✅ Pipeline saved to: {model_path}")
print(f"📦 File size: {os.path.getsize(model_path) / 1024:.1f} KB")

# Save metadata
model_metadata = {
    'model_name': 'Telecom Customer Churn Predictor',
    'version': '1.0.0',
    'created_at': datetime.now().isoformat(),
    'algorithm': 'GradientBoostingClassifier',
    'features': {
        'numeric': numeric_features_full,
        'categorical': categorical_features_full
    },
    'performance': {
        'cv_auc_mean': float(cv_scores.mean()),
        'cv_auc_std': float(cv_scores.std())
    },
    'preprocessing': {
        'numeric_strategy': 'median_imputation + standard_scaling',
        'categorical_strategy': 'constant_imputation + one_hot_encoding'
    },
    'target_variable': 'Churn (1=Yes, 0=No)',
    'model_format': 'joblib'
}

with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)

print(f"✅ Metadata saved to: {metadata_path}")
print(f"\n📋 Model Metadata:")
print(json.dumps(model_metadata, indent=2))

In [ ]:
# 🔄 Test Loading the Model
print("🔄 Testing model deserialization...")
loaded_pipeline = joblib.load(model_path)
print("✅ Model loaded successfully!")

# Verify it works
test_pred = loaded_pipeline.predict_proba(X_full.head(5))[:, 1]
print(f"📊 Sample predictions from loaded model: {test_pred.round(3)}")
print("✅ Deserialization verification complete!")

## 🚀 Creating an Inference Function for New Customers

In [ ]:
# 🚀 Production Inference Function
def predict_churn_risk(new_customer_data, model_path='./models/churn_prediction_pipeline.joblib', 
                       return_shap=False, shap_explainer=None):
    """
    Production-ready inference function for churn prediction.
    
    Parameters:
    -----------
    new_customer_data : pd.DataFrame
        Customer data with same features as training data
    model_path : str
        Path to saved model
    return_shap : bool
        Whether to return SHAP values for interpretability
    shap_explainer : shap.TreeExplainer, optional
        Pre-fitted SHAP explainer
        
    Returns:
    --------
    dict : Prediction results with probability, class, and optional SHAP explanation
    """
    # Load model
    model = joblib.load(model_path)
    
    # Ensure DataFrame format
    if isinstance(new_customer_data, dict):
        new_customer_data = pd.DataFrame([new_customer_data])
    
    # Predict
    churn_proba = model.predict_proba(new_customer_data)[:, 1]
    churn_pred = (churn_proba > 0.5).astype(int)
    
    results = []
    for i, (prob, pred) in enumerate(zip(churn_proba, churn_pred)):
        result = {
            'customer_id': new_customer_data.index[i] if len(new_customer_data) > 1 else 0,
            'churn_probability': float(prob),
            'churn_prediction': 'Yes' if pred == 1 else 'No',
            'risk_level': 'Critical' if prob > 0.8 else 'High' if prob > 0.6 else 'Medium' if prob > 0.4 else 'Low',
            'confidence': 'High' if abs(prob - 0.5) > 0.3 else 'Medium' if abs(prob - 0.5) > 0.15 else 'Low',
            'timestamp': datetime.now().isoformat()
        }
        
        # Add SHAP explanation if requested
        if return_shap and shap_explainer is not None:
            try:
                # Get preprocessed data for SHAP
                preprocessor = model.named_steps['preprocessor']
                X_proc = preprocessor.transform(new_customer_data.iloc[[i]])
                shap_vals = shap_explainer.shap_values(X_proc)
                
                # Get feature names
                feature_names = (numeric_features_full + 
                               list(preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features_full)))
                
                # Create explanation
                shap_df = pd.DataFrame({
                    'feature': feature_names,
                    'shap_value': shap_vals[1][0] if isinstance(shap_vals, list) else shap_vals[0]
                }).sort_values('shap_value', key=abs, ascending=False)
                
                result['top_risk_factors'] = shap_df.head(5).to_dict('records')
            except Exception as e:
                result['shap_error'] = str(e)
        
        results.append(result)
    
    return results[0] if len(results) == 1 else results

# 🧪 Test with a sample new customer
sample_new_customer = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 2,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 95.00,
    'TotalCharges': 190.00
}

print("🧪 Testing Inference Function with Sample Customer:")
print("="*60)
prediction = predict_churn_risk(sample_new_customer, model_path=model_path)
print(f"\n📊 Prediction Result:")
print(json.dumps(prediction, indent=2))

In [ ]:
# 🏢 Batch Inference for Multiple New Customers
def batch_predict(new_customers_df, model_path='./models/churn_prediction_pipeline.joblib'):
    """Process batch of customers efficiently."""
    predictions = predict_churn_risk(new_customers_df, model_path)
    return pd.DataFrame(predictions)

# Create sample batch
batch_customers = pd.DataFrame([
    {**sample_new_customer, 'customer_id': 'CUST_NEW_001'},
    {**sample_new_customer, 'tenure': 48, 'Contract': 'Two year', 'customer_id': 'CUST_NEW_002'},
    {**sample_new_customer, 'tenure': 12, 'Contract': 'One year', 'PaymentMethod': 'Credit card (automatic)', 'customer_id': 'CUST_NEW_003'}
]).set_index('customer_id')

print("🏢 Batch Inference Results:")
print("="*80)
batch_results = batch_predict(batch_customers, model_path)
print(batch_results[['churn_probability', 'churn_prediction', 'risk_level', 'confidence']].to_string())

## 📈 Basic Model Monitoring & Logging Concepts

Monitoring ensures our model stays healthy in production. Let's implement logging and basic monitoring!

In [ ]:
# 📈 Setup Production Logging
def setup_logger(name='churn_model', log_file='./models/churn_model.log'):
    """Setup structured logging for model monitoring."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    
    # Prevent duplicate handlers
    if not logger.handlers:
        # File handler
        fh = logging.FileHandler(log_file)
        fh.setLevel(logging.INFO)
        
        # Console handler
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        
        # Formatter
        formatter = logging.Formatter(
            '%(asctime)s | %(name)s | %(levelname)s | %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        )
        fh.setFormatter(formatter)
        ch.setFormatter(formatter)
        
        logger.addHandler(fh)
        logger.addHandler(ch)
    
    return logger

# Initialize logger
logger = setup_logger()
logger.info("🚀 Churn Prediction Model Logger Initialized")
print("✅ Logger configured successfully!")

In [ ]:
# 📊 Model Monitoring Class
class ChurnModelMonitor:
    """Basic monitoring system for churn prediction model."""
    
    def __init__(self, model_path, reference_data=None, logger=None):
        self.model_path = model_path
        self.model = joblib.load(model_path)
        self.reference_data = reference_data
        self.logger = logger or logging.getLogger('churn_model')
        self.prediction_history = []
        self.drift_history = []
        
    def log_prediction(self, customer_data, prediction_result):
        """Log each prediction for audit trail."""
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'prediction': prediction_result['churn_probability'],
            'risk_level': prediction_result['risk_level'],
            'input_features_hash': hash(str(customer_data.values.tobytes()))
        }
        self.prediction_history.append(log_entry)
        self.logger.info(f"Prediction logged: Risk={prediction_result['risk_level']}, Prob={prediction_result['churn_probability']:.3f}")
        return log_entry
    
    def check_data_drift(self, new_data, threshold=0.1):
        """Basic data drift detection using feature statistics."""
        if self.reference_data is None:
            self.logger.warning("No reference data provided for drift detection")
            return {'status': 'unknown', 'message': 'No reference data'}
        
        drift_report = {}
        numeric_cols = new_data.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            if col in self.reference_data.columns:
                ref_mean = self.reference_data[col].mean()
                new_mean = new_data[col].mean()
                drift = abs(new_mean - ref_mean) / (abs(ref_mean) + 1e-10)
                
                drift_report[col] = {
                    'reference_mean': float(ref_mean),
                    'new_mean': float(new_mean),
                    'relative_drift': float(drift),
                    'alert': drift > threshold
                }
        
        alerts = [col for col, stats in drift_report.items() if stats['alert']]
        status = 'alert' if alerts else 'healthy'
        
        self.logger.info(f"Drift check: {status.upper()} | Alerts: {len(alerts)}")
        
        return {
            'status': status,
            'alerts': alerts,
            'drift_details': drift_report,
            'checked_at': datetime.now().isoformat()
        }
    
    def get_prediction_stats(self, last_n=100):
        """Get statistics on recent predictions."""
        if not self.prediction_history:
            return {'status': 'no_data'}
        
        recent = self.prediction_history[-last_n:]
        probs = [p['prediction'] for p in recent]
        
        return {
            'total_predictions': len(self.prediction_history),
            'recent_predictions': len(recent),
            'mean_churn_probability': np.mean(probs),
            'high_risk_rate': sum(1 for p in recent if p['risk_level'] in ['High', 'Critical']) / len(recent),
            'timestamp': datetime.now().isoformat()
        }
    
    def health_check(self):
        """Perform system health check."""
        checks = {
            'model_loaded': self.model is not None,
            'model_path_exists': os.path.exists(self.model_path),
            'prediction_history_size': len(self.prediction_history),
            'timestamp': datetime.now().isoformat()
        }
        checks['overall_status'] = 'healthy' if all([checks['model_loaded'], checks['model_path_exists']]) else 'unhealthy'
        return checks

# Initialize monitor
monitor = ChurnModelMonitor(
    model_path=model_path,
    reference_data=X_full,
    logger=logger
)

print("✅ Model Monitor initialized!")
print(f"📊 Health Check: {monitor.health_check()}")

In [ ]:
# 🧪 Test Monitoring System
print("🧪 Testing Monitoring System")
print("="*60)

# Simulate predictions
for i in range(5):
    sample = X_test.iloc[[i]]
    pred = predict_churn_risk(sample, model_path)
    monitor.log_prediction(sample, pred)
    print(f"   Prediction {i+1}: Risk={pred['risk_level']}, Prob={pred['churn_probability']:.2%}")

# Check prediction stats
print(f"\n📊 Prediction Statistics:")
stats = monitor.get_prediction_stats()
print(json.dumps(stats, indent=2))

# Check for drift with test data
print(f"\n🔍 Data Drift Check (Test vs Training):")
drift = monitor.check_data_drift(X_test, threshold=0.15)
print(f"Status: {drift['status'].upper()}")
if drift['alerts']:
    print(f"⚠️  Alerting features: {drift['alerts']}")
else:
    print("✅ No significant drift detected")

In [ ]:
# 📈 Create Monitoring Dashboard Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Prediction Distribution
probs = best_pipeline.predict_proba(X_test)[:, 1]
axes[0, 0].hist(probs, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(x=0.5, color='red', linestyle='--', label='Decision Threshold')
axes[0, 0].set_title('📊 Churn Probability Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Churn Probability')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

# 2. Risk Level Breakdown
risk_levels = pd.cut(probs, bins=[0, 0.4, 0.6, 0.8, 1.0], 
                     labels=['Low', 'Medium', 'High', 'Critical'])
risk_counts = risk_levels.value_counts()
colors = ['green', 'yellow', 'orange', 'red']
axes[0, 1].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%', 
               colors=colors, startangle=90)
axes[0, 1].set_title('🎯 Risk Level Distribution', fontweight='bold')

# 3. Feature Drift (if reference available)
drift_data = monitor.check_data_drift(X_test.sample(500, random_state=42), threshold=0.15)
if 'drift_details' in drift_data:
    drift_values = [v['relative_drift'] for v in drift_data['drift_details'].values()]
    drift_names = list(drift_data['drift_details'].keys())
    colors_drift = ['red' if v > 0.15 else 'green' for v in drift_values]
    axes[1, 0].barh(drift_names, drift_values, color=colors_drift)
    axes[1, 0].axvline(x=0.15, color='red', linestyle='--', label='Alert Threshold')
    axes[1, 0].set_title('🔍 Feature Drift Detection', fontweight='bold')
    axes[1, 0].set_xlabel('Relative Drift')
    axes[1, 0].legend()

# 4. Model Performance Over Time (simulated)
dates = pd.date_range(end=datetime.now(), periods=30, freq='D')
np.random.seed(42)
auc_trend = 0.85 + np.cumsum(np.random.normal(0, 0.005, 30))
axes[1, 1].plot(dates, auc_trend, marker='o', color='purple', linewidth=2)
axes[1, 1].axhline(y=0.80, color='red', linestyle='--', label='Minimum Acceptable AUC')
axes[1, 1].fill_between(dates, 0.80, auc_trend, where=(auc_trend > 0.80), alpha=0.3, color='green')
axes[1, 1].set_title('📈 Model Performance Trend (Simulated)', fontweight='bold')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('AUC Score')
axes[1, 1].legend()
axes[1, 1].tick_params(axis='x', rotation=45)

plt.suptitle('📊 Production Model Monitoring Dashboard', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("✅ Monitoring dashboard generated!")
print("💡 In production, these metrics would be streamed to Grafana/Datadog/AWS CloudWatch")

## 🛠️ Hands-On Exercises

Time to put your skills to the test! Complete these challenges to solidify your understanding.


### 🎯 Exercise 1: SHAP Dependence Plot
Create a SHAP dependence plot for the most important feature to understand its interaction with a second feature. Analyze how the relationship changes across different feature values.

**Hint**: Use `shap.dependence_plot()` with `interaction_index` parameter.


### 🎯 Exercise 2: Custom Business Rules Layer
Build a wrapper around the prediction function that applies business rules AFTER the ML prediction. For example:
- If customer is a "SeniorCitizen" with tenure > 60 months, override to "Low Risk" regardless of model output
- If monthly charges > $100 AND contract is Month-to-month, add +10% to churn probability
- Log all overrides for compliance auditing


### 🎯 Exercise 3: A/B Testing Framework Simulation
Simulate an A/B test where:
- Group A gets predictions from the current Gradient Boosting model
- Group B gets predictions from a simpler Logistic Regression model
- Compare their precision, recall, and business value (assume each correct churn prediction saves $500, each false positive costs $50)
- Which model would you deploy and why?


### 🎯 Exercise 4: Automated Retraining Trigger
Implement a function that:
- Monitors model performance on new incoming data weekly
- Triggers a retraining alert if AUC drops below 0.80 for 2 consecutive weeks
- Saves a backup of the old model before retraining
- Logs the retraining event with timestamp and performance delta


## ✅ Solutions

Here are complete, production-ready solutions to all exercises. Study them carefully and adapt them to your use cases!


In [ ]:
# ✅ SOLUTION 1: SHAP Dependence Plot with Interaction
print("🎯 SOLUTION 1: SHAP Dependence Plot")
print("="*60)

# Get the most important feature
top_feature = shap_importance.iloc[0]['feature']
print(f"📊 Analyzing dependence for top feature: {top_feature}")

# Create dependence plot
plt.figure(figsize=(12, 6))
shap.dependence_plot(
    top_feature,
    shap_values[1] if isinstance(shap_values, list) else shap_values,
    X_test_processed_df,
    feature_names=feature_names,
    show=False
)
plt.title(f'🔍 SHAP Dependence Plot: {top_feature}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Interpretation:")
print("   • Each point = one customer")
print("   • X-axis = feature value")
print("   • Y-axis = SHAP value (impact on prediction)")
print("   • Color = value of interacting feature (auto-selected)")
print("   • Vertical dispersion indicates interaction effects")

In [ ]:
# ✅ SOLUTION 2: Custom Business Rules Layer
print("🏢 SOLUTION 2: Business Rules Engine")
print("="*60)

class BusinessRulesEngine:
    """Applies business logic on top of ML predictions."""
    
    def __init__(self, base_model_path, logger=None):
        self.model_path = base_model_path
        self.model = joblib.load(base_model_path)
        self.logger = logger or logging.getLogger('business_rules')
        self.override_log = []
    
    def predict_with_rules(self, customer_data):
        """Apply ML prediction + business rules."""
        # Base ML prediction
        base_pred = predict_churn_risk(customer_data, self.model_path)
        base_prob = base_pred['churn_probability']
        
        # Initialize with ML prediction
        final_prob = base_prob
        final_risk = base_pred['risk_level']
        rules_applied = []
        
        # Rule 1: Senior Citizen Loyalty Override
        if 'SeniorCitizen' in customer_data.columns and 'tenure' in customer_data.columns:
            if customer_data['SeniorCitizen'].iloc[0] == 1 and customer_data['tenure'].iloc[0] > 60:
                final_prob = max(0.1, base_prob - 0.3)  # Cap at 10% minimum
                final_risk = 'Low'
                rules_applied.append("Senior Loyalty Override: -30% risk")
        
        # Rule 2: High-Value Month-to-Month Risk Boost
        if 'MonthlyCharges' in customer_data.columns and 'Contract' in customer_data.columns:
            if (customer_data['MonthlyCharges'].iloc[0] > 100 and 
                customer_data['Contract'].iloc[0] == 'Month-to-month'):
                final_prob = min(0.95, base_prob + 0.10)
                final_risk = 'High' if final_prob > 0.6 else 'Medium'
                rules_applied.append("High-Value Month-to-Month Boost: +10% risk")
        
        # Rule 3: Recent Complaint Escalation (simulated)
        if 'TechSupport' in customer_data.columns:
            if customer_data['TechSupport'].iloc[0] == 'No':
                final_prob = min(0.95, final_prob + 0.05)
                rules_applied.append("No TechSupport Flag: +5% risk")
        
        # Log override
        if rules_applied:
            override_record = {
                'timestamp': datetime.now().isoformat(),
                'base_probability': float(base_prob),
                'final_probability': float(final_prob),
                'rules_applied': rules_applied,
                'customer_segment': self._get_segment(customer_data)
            }
            self.override_log.append(override_record)
            self.logger.info(f"Business rules applied: {rules_applied}")
        
        return {
            'base_prediction': base_pred,
            'final_churn_probability': float(final_prob),
            'final_risk_level': final_risk,
            'rules_applied': rules_applied,
            'override_occurred': len(rules_applied) > 0
        }
    
    def _get_segment(self, customer_data):
        """Determine customer segment for reporting."""
        if 'tenure' in customer_data.columns:
            tenure = customer_data['tenure'].iloc[0]
            return 'New' if tenure <= 12 else 'Established' if tenure <= 36 else 'Loyal'
        return 'Unknown'
    
    def get_override_summary(self):
        """Summary of all rule overrides for compliance."""
        return {
            'total_overrides': len(self.override_log),
            'override_rate': len(self.override_log) / max(len(self.override_log), 1),
            'recent_overrides': self.override_log[-10:],
            'rule_frequency': self._count_rule_frequency()
        }
    
    def _count_rule_frequency(self):
        """Count how often each rule is triggered."""
        from collections import Counter
        all_rules = []
        for record in self.override_log:
            all_rules.extend(record['rules_applied'])
        return dict(Counter(all_rules))

# Test the business rules engine
rules_engine = BusinessRulesEngine(model_path, logger)

# Test case 1: Senior citizen with long tenure (should get override)
senior_loyal = pd.DataFrame([{
    'gender': 'Male', 'SeniorCitizen': 1, 'Partner': 'Yes', 'Dependents': 'Yes',
    'tenure': 65, 'PhoneService': 'Yes', 'MultipleLines': 'Yes',
    'InternetService': 'DSL', 'OnlineSecurity': 'Yes', 'OnlineBackup': 'Yes',
    'DeviceProtection': 'Yes', 'TechSupport': 'Yes', 'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes', 'Contract': 'Two year', 'PaperlessBilling': 'No',
    'PaymentMethod': 'Bank transfer (automatic)', 'MonthlyCharges': 65.00,
    'TotalCharges': 4500.00
}])

result1 = rules_engine.predict_with_rules(senior_loyal)
print("\n🔴 Test Case 1 - Senior Loyal Customer:")
print(f"   Base Probability: {result1['base_prediction']['churn_probability']:.1%}")
print(f"   Final Probability: {result1['final_churn_probability']:.1%}")
print(f"   Rules Applied: {result1['rules_applied']}")
print(f"   Override: {result1['override_occurred']}")

# Test case 2: High-value month-to-month
high_value_mtm = pd.DataFrame([{
    'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'No', 'Dependents': 'No',
    'tenure': 8, 'PhoneService': 'Yes', 'MultipleLines': 'Yes',
    'InternetService': 'Fiber optic', 'OnlineSecurity': 'No', 'OnlineBackup': 'No',
    'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes', 'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check', 'MonthlyCharges': 105.00,
    'TotalCharges': 840.00
}])

result2 = rules_engine.predict_with_rules(high_value_mtm)
print("\n🟠 Test Case 2 - High-Value Month-to-Month:")
print(f"   Base Probability: {result2['base_prediction']['churn_probability']:.1%}")
print(f"   Final Probability: {result2['final_churn_probability']:.1%}")
print(f"   Rules Applied: {result2['rules_applied']}")
print(f"   Override: {result2['override_occurred']}")

print("\n📊 Override Summary:")
print(json.dumps(rules_engine.get_override_summary(), indent=2))

In [ ]:
# ✅ SOLUTION 3: A/B Testing Framework Simulation
print("🧪 SOLUTION 3: A/B Testing Framework")
print("="*60)

class ABTestFramework:
    """Simulate A/B test between two models."""
    
    def __init__(self, model_a, model_b, test_data, true_labels, 
                 cost_fp=50, cost_fn=500, revenue_tp=500):
        self.model_a = model_a
        self.model_b = model_b
        self.X_test = test_data
        self.y_true = true_labels
        self.cost_fp = cost_fp  # False Positive cost
        self.cost_fn = cost_fn  # False Negative cost (missed churn)
        self.revenue_tp = revenue_tp  # True Positive revenue (saved customer)
        
    def evaluate_model(self, model, name):
        """Evaluate model with business metrics."""
        y_pred = model.predict(self.X_test)
        y_prob = model.predict_proba(self.X_test)[:, 1]
        
        # Standard metrics
        from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
        
        precision = precision_score(self.y_true, y_pred)
        recall = recall_score(self.y_true, y_pred)
        f1 = f1_score(self.y_true, y_pred)
        accuracy = accuracy_score(self.y_true, y_pred)
        auc = roc_auc_score(self.y_true, y_prob)
        
        # Business metrics
        cm = confusion_matrix(self.y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        
        # Calculate business value
        total_value = (tp * self.revenue_tp) - (fp * self.cost_fp) - (fn * self.cost_fn)
        value_per_customer = total_value / len(self.y_true)
        
        return {
            'model_name': name,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'accuracy': accuracy,
            'auc': auc,
            'confusion_matrix': {'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)},
            'business_value': {
                'total_value': total_value,
                'value_per_customer': value_per_customer,
                'true_positives_revenue': tp * self.revenue_tp,
                'false_positives_cost': fp * self.cost_fp,
                'false_negatives_cost': fn * self.cost_fn
            }
        }
    
    def run_test(self):
        """Run full A/B test comparison."""
        print("🏃 Running A/B Test Simulation...")
        
        results_a = self.evaluate_model(self.model_a, "Model A: Gradient Boosting")
        results_b = self.evaluate_model(self.model_b, "Model B: Logistic Regression")
        
        return results_a, results_b
    
    def print_comparison(self, results_a, results_b):
        """Print formatted comparison."""
        print("\n" + "="*80)
        print("📊 A/B TEST RESULTS")
        print("="*80)
        
        metrics = ['precision', 'recall', 'f1_score', 'accuracy', 'auc']
        print(f"\n{'Metric':<25} {'Model A (GB)':<15} {'Model B (LR)':<15} {'Winner':<10}")
        print("-"*80)
        
        for metric in metrics:
            val_a = results_a[metric]
            val_b = results_b[metric]
            winner = "Model A" if val_a > val_b else "Model B" if val_b > val_a else "Tie"
            print(f"{metric.upper():<25} {val_a:<15.4f} {val_b:<15.4f} {winner:<10}")
        
        print("\n" + "-"*80)
        print("💰 BUSINESS VALUE ANALYSIS")
        print("-"*80)
        
        biz_a = results_a['business_value']
        biz_b = results_b['business_value']
        
        print(f"{'Metric':<30} {'Model A (GB)':<20} {'Model B (LR)':<20}")
        print("-"*80)
        print(f"{'Total Business Value':<30} ${biz_a['total_value']:>18,} ${biz_b['total_value']:>18,}")
        print(f"{'Value per Customer':<30} ${biz_a['value_per_customer']:>18.2f} ${biz_b['value_per_customer']:>18.2f}")
        print(f"{'TP Revenue (Saved Customers)':<30} ${biz_a['true_positives_revenue']:>18,} ${biz_b['true_positives_revenue']:>18,}")
        print(f"{'FP Cost (Wasted Retention)':<30} ${biz_a['false_positives_cost']:>18,} ${biz_b['false_positives_cost']:>18,}")
        print(f"{'FN Cost (Missed Churn)':<30} ${biz_a['false_negatives_cost']:>18,} ${biz_b['false_negatives_cost']:>18,}")
        
        # Recommendation
        winner = results_a if biz_a['total_value'] > biz_b['total_value'] else results_b
        print("\n" + "="*80)
        print(f"🏆 RECOMMENDATION: Deploy {winner['model_name']}")
        print(f"💡 Reason: Higher business value (${winner['business_value']['total_value']:,})")
        print("="*80)

# Create Model B: Logistic Regression with same preprocessing
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),  # Reuse same preprocessor
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
)
])

# Train both models on same data
print("🚀 Training both models for A/B test...")
gb_model = best_pipeline  # Already trained
lr_model = lr_pipeline.fit(X_train, y_train)

# Run A/B test
ab_test = ABTestFramework(
    model_a=gb_model,
    model_b=lr_model,
    test_data=X_test,
    true_labels=y_test,
    cost_fp=50,
    cost_fn=500,
    revenue_tp=500
)

results_a, results_b = ab_test.run_test()
ab_test.print_comparison(results_a, results_b)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metric comparison
metrics = ['precision', 'recall', 'f1_score', 'auc']
a_scores = [results_a[m] for m in metrics]
b_scores = [results_b[m] for m in metrics]
x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, a_scores, width, label='Model A (GB)', color='steelblue')
axes[0].bar(x + width/2, b_scores, width, label='Model B (LR)', color='coral')
axes[0].set_ylabel('Score')
axes[0].set_title('📊 Model Performance Comparison', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels([m.upper() for m in metrics])
axes[0].legend()
axes[0].set_ylim(0, 1)

# Business value comparison
biz_metrics = ['Total Value', 'TP Revenue', 'FP Cost', 'FN Cost']
a_biz = [results_a['business_value']['total_value'], 
         results_a['business_value']['true_positives_revenue'],
         results_a['business_value']['false_positives_cost'],
         results_a['business_value']['false_negatives_cost']]
b_biz = [results_b['business_value']['total_value'],
         results_b['business_value']['true_positives_revenue'],
         results_b['business_value']['false_positives_cost'],
         results_b['business_value']['false_negatives_cost']]

axes[1].bar(x - width/2, a_biz, width, label='Model A (GB)', color='steelblue')
axes[1].bar(x + width/2, b_biz, width, label='Model B (LR)', color='coral')
axes[1].set_ylabel('USD ($)')
axes[1].set_title('💰 Business Value Comparison', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(biz_metrics, rotation=15, ha='right')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ✅ SOLUTION 4: Automated Retraining Trigger
print("🔄 SOLUTION 4: Automated Retraining System")
print("="*60)

class AutoRetrainingSystem:
    """Monitors model performance and triggers retraining when needed."""
    
    def __init__(self, model_path, training_data, target_data, 
                 min_auc_threshold=0.80, trigger_consecutive_weeks=2,
                 backup_dir='./model_backups'):
        self.model_path = model_path
        self.training_data = training_data
        self.target_data = target_data
        self.min_auc = min_auc_threshold
        self.trigger_weeks = trigger_consecutive_weeks
        self.backup_dir = backup_dir
        self.performance_history = []
        self.retraining_log = []
        
        os.makedirs(backup_dir, exist_ok=True)
        
    def evaluate_weekly_performance(self, new_data, new_targets, week_id):
        """Evaluate model on new week's data."""
        model = joblib.load(self.model_path)
        
        # Calculate AUC
        y_prob = model.predict_proba(new_data)[:, 1]
        auc = roc_auc_score(new_targets, y_prob)
        
        # Calculate additional metrics
        y_pred = model.predict(new_data)
        precision = precision_score(new_targets, y_pred, zero_division=0)
        recall = recall_score(new_targets, y_pred, zero_division=0)
        
        record = {
            'week_id': week_id,
            'timestamp': datetime.now().isoformat(),
            'auc': float(auc),
            'precision': float(precision),
            'recall': float(recall),
            'samples_evaluated': len(new_data)
        }
        
        self.performance_history.append(record)
        return record
    
    def check_retraining_needed(self):
        """Check if retraining should be triggered."""
        if len(self.performance_history) < self.trigger_weeks:
            return {'status': 'insufficient_data', 'message': f'Need {self.trigger_weeks} weeks of data'}
        
        # Check last N weeks
        recent = self.performance_history[-self.trigger_weeks:]
        below_threshold = [w for w in recent if w['auc'] < self.min_auc]
        
        if len(below_threshold) >= self.trigger_weeks:
            return {
                'status': 'trigger_retraining',
                'reason': f'AUC below {self.min_auc} for {self.trigger_weeks} consecutive weeks',
                'affected_weeks': [w['week_id'] for w in below_threshold],
                'auc_values': [w['auc'] for w in below_threshold]
            }
        
        return {
            'status': 'healthy',
            'recent_auc': [w['auc'] for w in recent],
            'message': 'Model performance within acceptable range'
        }
    
    def backup_current_model(self):
        """Create timestamped backup of current model."""
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        backup_path = os.path.join(self.backup_dir, f'model_backup_{timestamp}.joblib')
        
        current_model = joblib.load(self.model_path)
        joblib.dump(current_model, backup_path)
        
        return {'backup_path': backup_path, 'timestamp': timestamp}
    
    def retrain_model(self):
        """Retrain model on full historical + new data."""
        print("🔄 Initiating model retraining...")
        
        # Backup old model
        backup_info = self.backup_current_model()
        print(f"💾 Old model backed up to: {backup_info['backup_path']}")
        
        # Retrain (in practice, you'd combine old + new data)
        new_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', GradientBoostingClassifier(
                n_estimators=200, learning_rate=0.1, max_depth=4,
                min_samples_split=20, min_samples_leaf=10,
                subsample=0.8, random_state=42
            ))
        ])
        
        # Simulate retraining on combined data
        new_pipeline.fit(self.training_data, self.target_data)
        
        # Save new model
        joblib.dump(new_pipeline, self.model_path)
        
        # Evaluate new model
        new_auc = roc_auc_score(
            self.target_data, 
            new_pipeline.predict_proba(self.training_data)[:, 1]
        )
        
        retraining_record = {
            'timestamp': datetime.now().isoformat(),
            'backup_path': backup_info['backup_path'],
            'previous_auc': self.performance_history[-1]['auc'] if self.performance_history else None,
            'new_auc': float(new_auc),
            'auc_delta': float(new_auc - (self.performance_history[-1]['auc'] if self.performance_history else 0)),
            'trigger_reason': self.check_retraining_needed().get('reason', 'manual')
        }
        
        self.retraining_log.append(retraining_record)
        print(f"✅ Retraining complete! New AUC: {new_auc:.4f}")
        
        return retraining_record
    
    def run_weekly_check(self, new_week_data, new_week_targets, week_id):
        """Full weekly monitoring workflow."""
        print(f"\n📅 Running Week {week_id} Check...")
        
        # Evaluate
        perf = self.evaluate_weekly_performance(new_week_data, new_week_targets, week_id)
        print(f"   AUC: {perf['auc']:.4f} | Precision: {perf['precision']:.4f} | Recall: {perf['recall']:.4f}")
        
        # Check if retraining needed
        status = self.check_retraining_needed()
        print(f"   Status: {status['status'].upper()}")
        
        if status['status'] == 'trigger_retraining':
            print(f"   ⚠️  ALERT: {status['reason']}")
            retrain_result = self.retrain_model()
            return {'action': 'retrained', 'details': retrain_result}
        
        return {'action': 'monitoring', 'status': status}
    
    def get_performance_trend(self):
        """Get performance history for visualization."""
        return pd.DataFrame(self.performance_history)

# 🧪 Simulate 8 weeks of monitoring
print("🧪 Simulating 8-week monitoring period...")
print("(Weeks 5-6 will simulate performance degradation)")
print("="*60)

# Initialize system
auto_system = AutoRetrainingSystem(
    model_path=model_path,
    training_data=X_train,
    target_data=y_train,
    min_auc_threshold=0.80,
    trigger_consecutive_weeks=2
)

# Simulate weeks
np.random.seed(42)
for week in range(1, 9):
    # Simulate different data distributions
    if week in [5, 6]:
        # Simulate degraded performance (concept drift)
        week_data = X_test.sample(300, random_state=week).copy()
        # Add noise to features to simulate drift
        for col in week_data.select_dtypes(include=[np.number]).columns:
            week_data[col] = week_data[col] * np.random.uniform(0.7, 1.3, size=len(week_data))
        week_targets = y_test.loc[week_data.index]
    else:
        week_data = X_test.sample(300, random_state=week)
        week_targets = y_test.loc[week_data.index]
    
    result = auto_system.run_weekly_check(week_data, week_targets, week)
    
    if result['action'] == 'retrained':
        print(f"\n🎉 Model automatically retrained due to performance degradation!")
        print(f"   AUC improvement: {result['details']['auc_delta']:+.4f}")

# Visualize performance trend
trend_df = auto_system.get_performance_trend()

plt.figure(figsize=(12, 6))
plt.plot(trend_df['week_id'], trend_df['auc'], marker='o', linewidth=2, markersize=8, label='AUC Score')
plt.axhline(y=0.80, color='red', linestyle='--', linewidth=2, label='Retraining Threshold')
plt.fill_between(trend_df['week_id'], 0.75, 0.80, alpha=0.2, color='red', label='Danger Zone')

# Mark retraining events
for log in auto_system.retraining_log:
    # Find approximate week from timestamp
    plt.axvline(x=5.5, color='green', linestyle=':', linewidth=2, alpha=0.7, label='Retraining Event')
    break  # Only show once in legend

plt.title('📈 8-Week Model Performance Monitoring with Auto-Retraining', fontsize=14, fontweight='bold')
plt.xlabel('Week')
plt.ylabel('AUC Score')
plt.ylim(0.70, 1.0)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Performance Trend Summary:")
print(trend_df[['week_id', 'auc', 'precision', 'recall']].to_string(index=False))

if auto_system.retraining_log:
    print(f"\n🔄 Retraining Events Logged: {len(auto_system.retraining_log)}")
    for log in auto_system.retraining_log:
        print(f"   • {log['timestamp']}: AUC delta {log['auc_delta']:+.4f}")
else:
    print("\n✅ No retraining needed — model remained stable!")